# Model Tuning — Cari Akurasi Lebih Tinggi Tanpa Overfit

**Goal:** Improve R² > 0.62 (baseline) sambil keep train-test gap < 0.1.

**Strategi yang dicoba:**
1. Feature selection (buang yang tidak signifikan)
2. Ridge Regression dengan tuning lambda via CV
3. Lasso (L1) untuk auto feature selection
4. Add nonlinear features (interaksi lag1*daily_change)
5. Time-series cross-validation (TSCV)

**Anti-overfit checks:**
- Test R² tidak boleh < Train R² - 0.15
- TSCV avg R² jadi metrik utama (lebih robust)
- Feature count minimum

## 1. Setup & Load

In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

MS_PER_DAY = 86400000

with open('../honda-dummy.json', 'r') as f:
    data = json.load(f)
inventory = data['inventory']
transactions = data['transactions']

def build_daily_series(transactions, current_quantity):
    daily_delta = {}
    for tx in transactions:
        ts = int(tx['timestamp'])
        qty = int(tx['quantity'])
        tx_type = tx.get('type', 'out')
        day_key = (ts // MS_PER_DAY) * MS_PER_DAY
        signed_qty = -abs(qty) if tx_type == 'out' else abs(qty)
        daily_delta[day_key] = daily_delta.get(day_key, 0) + signed_qty
    days = sorted(daily_delta.keys())
    if not days: return pd.DataFrame()
    total_delta = sum(daily_delta.values())
    level = current_quantity - total_delta
    rows = []
    for day in days:
        level += daily_delta[day]
        rows.append({'date': pd.to_datetime(day, unit='ms'), 'quantity': max(0, level)})
    return pd.DataFrame(rows)

tx_per_item = {}
for tx_id, tx in transactions.items():
    tx_per_item.setdefault(tx['itemId'], []).append(tx)

series_per_item = {}
for item_id, item in inventory.items():
    df = build_daily_series(tx_per_item.get(item_id, []), item['quantity'])
    if len(df) >= 30:
        series_per_item[item_id] = df

print(f'Items dengan ≥30 hari: {len(series_per_item)}')

Items dengan ≥30 hari: 20


## 2. Build Features (Sama dengan Production)

In [2]:
def build_features_v1(df):
    """9 features baseline (sama dengan production)."""
    df = df.copy().reset_index(drop=True)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['day_number'] = range(len(df))
    df['lag1'] = df['quantity'].shift(1)
    df['lag3'] = df['quantity'].shift(3)
    df['lag7'] = df['quantity'].shift(7)
    df['rolling_mean_7'] = df['quantity'].shift(1).rolling(7).mean()
    df['rolling_std_7'] = df['quantity'].shift(1).rolling(7).std()
    df['daily_change'] = df['quantity'].diff()
    df['target'] = df['quantity'].shift(-1)
    return df.dropna().reset_index(drop=True)

def build_features_v2(df):
    """Reduced + nonlinear features (anti-overfit, focus pada signifikan)."""
    df = df.copy().reset_index(drop=True)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['lag1'] = df['quantity'].shift(1)
    df['lag3'] = df['quantity'].shift(3)
    df['rolling_mean_7'] = df['quantity'].shift(1).rolling(7).mean()
    df['rolling_std_7'] = df['quantity'].shift(1).rolling(7).std()
    df['daily_change'] = df['quantity'].diff()
    df['lag1_x_dchange'] = df['lag1'] * df['daily_change']  # interaction
    df['target'] = df['quantity'].shift(-1)
    return df.dropna().reset_index(drop=True)

def build_features_v3(df):
    """Minimalis: cuma fitur signifikan dari OLS summary sebelumnya."""
    df = df.copy().reset_index(drop=True)
    df['lag1'] = df['quantity'].shift(1)
    df['daily_change'] = df['quantity'].diff()
    df['rolling_mean_7'] = df['quantity'].shift(1).rolling(7).mean()
    df['target'] = df['quantity'].shift(-1)
    return df.dropna().reset_index(drop=True)

FEATS = {
    'v1_full_9feat': ['lag1', 'lag3', 'lag7', 'rolling_mean_7', 'rolling_std_7', 'daily_change', 'day_of_week', 'is_weekend', 'day_number'],
    'v2_reduced_8feat': ['lag1', 'lag3', 'rolling_mean_7', 'rolling_std_7', 'daily_change', 'day_of_week', 'is_weekend', 'lag1_x_dchange'],
    'v3_minimal_3feat': ['lag1', 'daily_change', 'rolling_mean_7'],
}
BUILDERS = {'v1_full_9feat': build_features_v1, 'v2_reduced_8feat': build_features_v2, 'v3_minimal_3feat': build_features_v3}
print('Feature sets defined')

Feature sets defined


## 3. TSCV Eval Function

In [3]:
def evaluate_with_tscv(item_id, builder, feat_cols, model_class, model_kwargs=None, n_splits=5):
    """Time-series cross-validation untuk avoid overfitting bias."""
    df = series_per_item[item_id]
    if len(df) < 50:
        return None
    df_f = builder(df)
    if len(df_f) < 30:
        return None

    X = df_f[feat_cols].values
    y = df_f['target'].values

    tscv = TimeSeriesSplit(n_splits=n_splits)
    train_r2s, test_r2s = [], []
    for train_idx, test_idx in tscv.split(X):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_te_s = sc.transform(X_te)
        m = model_class(**(model_kwargs or {}))
        m.fit(X_tr_s, y_tr)
        train_r2s.append(r2_score(y_tr, m.predict(X_tr_s)))
        test_r2s.append(r2_score(y_te, m.predict(X_te_s)))

    return {
        'train_r2_mean': np.mean(train_r2s),
        'test_r2_mean': np.mean(test_r2s),
        'test_r2_std': np.std(test_r2s),
        'gap': np.mean(train_r2s) - np.mean(test_r2s),
    }

def run_experiment(name, builder_key, feat_cols, model_class, model_kwargs=None):
    rows = []
    for item_id in series_per_item:
        r = evaluate_with_tscv(item_id, BUILDERS[builder_key], feat_cols, model_class, model_kwargs)
        if r:
            rows.append({'item': inventory[item_id]['name'][:30], **r})
    df = pd.DataFrame(rows)
    return {
        'name': name,
        'n_items': len(df),
        'avg_test_r2': df['test_r2_mean'].mean(),
        'avg_train_r2': df['train_r2_mean'].mean(),
        'avg_gap': df['gap'].mean(),
        'pct_r2_above_05': (df['test_r2_mean'] > 0.5).sum() / len(df) * 100,
        'pct_r2_above_07': (df['test_r2_mean'] > 0.7).sum() / len(df) * 100,
        'overfit_items': (df['gap'] > 0.15).sum(),
        'detail': df,
    }

print('Eval functions ready')

Eval functions ready


## 4. Run All Experiments

Compare: 3 feature sets × 4 models = 12 combinations.

In [4]:
experiments = [
    # Baseline (production current)
    ('v1_OLS', 'v1_full_9feat', LinearRegression, None),
    ('v1_Ridge_lambda1', 'v1_full_9feat', Ridge, {'alpha': 1.0}),
    ('v1_Ridge_lambda10', 'v1_full_9feat', Ridge, {'alpha': 10.0}),
    ('v1_RidgeCV', 'v1_full_9feat', RidgeCV, {'alphas': [0.01, 0.1, 1.0, 10.0, 100.0]}),
    ('v1_Lasso', 'v1_full_9feat', Lasso, {'alpha': 0.1}),
    # Reduced features + interaksi
    ('v2_OLS', 'v2_reduced_8feat', LinearRegression, None),
    ('v2_Ridge_lambda1', 'v2_reduced_8feat', Ridge, {'alpha': 1.0}),
    ('v2_RidgeCV', 'v2_reduced_8feat', RidgeCV, {'alphas': [0.01, 0.1, 1.0, 10.0, 100.0]}),
    # Minimal
    ('v3_OLS', 'v3_minimal_3feat', LinearRegression, None),
    ('v3_Ridge_lambda1', 'v3_minimal_3feat', Ridge, {'alpha': 1.0}),
]

results = []
for name, fkey, model, kw in experiments:
    r = run_experiment(name, fkey, FEATS[fkey], model, kw)
    results.append({
        'Model': name,
        'Features': fkey,
        'N': r['n_items'],
        'Test R²': r['avg_test_r2'],
        'Train R²': r['avg_train_r2'],
        'Gap': r['avg_gap'],
        'R²>0.5': r['pct_r2_above_05'],
        'R²>0.7': r['pct_r2_above_07'],
        'Overfit': r['overfit_items'],
    })

df_results = pd.DataFrame(results).sort_values('Test R²', ascending=False)
print('\n=== HASIL EKSPERIMEN (sorted by Test R²) ===')
print(df_results.to_string(index=False))


=== HASIL EKSPERIMEN (sorted by Test R²) ===
            Model         Features  N  Test R²  Train R²      Gap  R²>0.5  R²>0.7  Overfit
           v3_OLS v3_minimal_3feat 20 0.620666  0.631357 0.010692   100.0    15.0        0
 v3_Ridge_lambda1 v3_minimal_3feat 20 0.620234  0.631009 0.010775   100.0    15.0        0
 v2_Ridge_lambda1 v2_reduced_8feat 20 0.609983  0.643476 0.033493    95.0    10.0        0
       v2_RidgeCV v2_reduced_8feat 20 0.607675  0.639771 0.032096    95.0    10.0        0
           v2_OLS v2_reduced_8feat 20 0.606309  0.644801 0.038492    95.0    10.0        0
         v1_Lasso    v1_full_9feat 20 0.598605  0.645468 0.046863    85.0    10.0        1
       v1_RidgeCV    v1_full_9feat 20 0.591673  0.642543 0.050870    85.0    10.0        1
 v1_Ridge_lambda1    v1_full_9feat 20 0.590715  0.647633 0.056918    85.0    10.0        2
v1_Ridge_lambda10    v1_full_9feat 20 0.582896  0.629725 0.046829    85.0     5.0        1
           v1_OLS    v1_full_9feat 20 0.5738

## 5. Pilih Pemenang

In [5]:
# Filter: gap < 0.15 AND test R² > baseline
good = df_results[(df_results['Gap'] < 0.15)].sort_values('Test R²', ascending=False)
print('=== TOP 5 (gap<0.15, no overfit) ===')
print(good.head(5).to_string(index=False))

winner = good.iloc[0]
print(f"\n🏆 Winner: {winner['Model']}")
print(f"   Test R²: {winner['Test R²']:.4f}")
print(f"   Train-Test Gap: {winner['Gap']:.4f}")
print(f"   Items dengan R² > 0.5: {winner['R²>0.5']:.0f}%")
print(f"   Items dengan R² > 0.7: {winner['R²>0.7']:.0f}%")

=== TOP 5 (gap<0.15, no overfit) ===
           Model         Features  N  Test R²  Train R²      Gap  R²>0.5  R²>0.7  Overfit
          v3_OLS v3_minimal_3feat 20 0.620666  0.631357 0.010692   100.0    15.0        0
v3_Ridge_lambda1 v3_minimal_3feat 20 0.620234  0.631009 0.010775   100.0    15.0        0
v2_Ridge_lambda1 v2_reduced_8feat 20 0.609983  0.643476 0.033493    95.0    10.0        0
      v2_RidgeCV v2_reduced_8feat 20 0.607675  0.639771 0.032096    95.0    10.0        0
          v2_OLS v2_reduced_8feat 20 0.606309  0.644801 0.038492    95.0    10.0        0

🏆 Winner: v3_OLS
   Test R²: 0.6207
   Train-Test Gap: 0.0107
   Items dengan R² > 0.5: 100%
   Items dengan R² > 0.7: 15%


## 6. Find Optimal Ridge Lambda

In [6]:
# Lambda search untuk Ridge dengan v1 features
lambdas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0]
lambda_results = []
for lam in lambdas:
    r = run_experiment(f'Ridge_lambda{lam}', 'v1_full_9feat', FEATS['v1_full_9feat'], Ridge, {'alpha': lam})
    lambda_results.append({
        'Lambda': lam,
        'Test R²': r['avg_test_r2'],
        'Train R²': r['avg_train_r2'],
        'Gap': r['avg_gap'],
    })
df_lam = pd.DataFrame(lambda_results)
print('=== Lambda search (v1 features) ===')
print(df_lam.to_string(index=False))

best_lam = df_lam.loc[df_lam['Test R²'].idxmax()]
print(f"\n🎯 Optimal lambda: {best_lam['Lambda']}")

=== Lambda search (v1 features) ===
 Lambda  Test R²  Train R²      Gap
  0.001 0.573971  0.649725 0.075754
  0.010 0.574670  0.649724 0.075054
  0.100 0.579774  0.649635 0.069861
  0.500 0.587908  0.648775 0.060867
  1.000 0.590715  0.647633 0.056918
  2.000 0.592054  0.645523 0.053469
  5.000 0.589984  0.639631 0.049647
 10.000 0.582896  0.629725 0.046829
 50.000 0.517262  0.554948 0.037686
100.000 0.451527  0.484930 0.033403

🎯 Optimal lambda: 2.0


## 7. Optimal Ridge Coefficients (untuk implement ke production)

In [7]:
# Ambil model terbaik dengan optimal lambda, train pakai semua data sample item
best_lambda = best_lam['Lambda']
sample_id = list(series_per_item.keys())[0]
df_sample = BUILDERS['v1_full_9feat'](series_per_item[sample_id])

X = df_sample[FEATS['v1_full_9feat']].values
y = df_sample['target'].values

sc = StandardScaler()
X_s = sc.fit_transform(X)

best_model = Ridge(alpha=best_lambda)
best_model.fit(X_s, y)

print(f"Best model: Ridge(alpha={best_lambda})")
print(f"Intercept: {best_model.intercept_:.4f}")
print(f"Coefficients:")
for name, coef in zip(FEATS['v1_full_9feat'], best_model.coef_):
    print(f"  {name:20s}: {coef:+.4f}")

Best model: Ridge(alpha=2.0)
Intercept: 70.6463
Coefficients:
  lag1                : +19.0472
  lag3                : -1.3218
  lag7                : -1.5502
  rolling_mean_7      : -1.6793
  rolling_std_7       : +3.4589
  daily_change        : +15.2996
  day_of_week         : -0.5469
  is_weekend          : +1.1948
  day_number          : -0.2500
